In [1]:
# Importing Libraries and connecting to Database
import pandas as pd
import sqlite3

DB_PATH = "../data/patents.db"
conn = sqlite3.connect(DB_PATH)

print("Connected to patents.db")

Connected to patents.db


In [2]:
#Q1 Top Inventors (Who has the most patents?)
q1 = """
SELECT 
    full_name,
    country,
    COUNT(DISTINCT patent_id) AS patent_count
FROM inventors
WHERE full_name IS NOT NULL
AND full_name != ''
GROUP BY inventor_id
ORDER BY patent_count DESC
LIMIT 20;
"""

df_q1 = pd.read_sql(q1, conn)
print("Q1: TOP INVENTORS")
print("")
print(df_q1.to_string(index=False))

Q1: TOP INVENTORS

              full_name country  patent_count
       Shunpei Yamazaki      JP            36
        Kia Silverbrook      AU            23
                Tao Luo      US            16
               Junyi Li      US            13
       Bartley K. Andre      US            12
  Matthew Dean Rohrbach      US            11
               Jing Sun      US            11
       Gurtej S. Sandhu      US            11
     Duncan Robert Kerr      US            11
       Roderick A. Hyde      US            10
      Rico Zörkendörfer      US            10
   Peter Russell-Clarke      US            10
          Kangguo Cheng      US            10
          David R. Hall      US            10
Christopher J. Stringer      US            10
          Xiaoxia Zhang      US             9
            Wanshi Chen      US             9
         Takashi Suzuki      JP             9
         Takahiro NISHI      JP             9
               Shan Liu      US             9


In [3]:
# Q2 Top Companies (Which companies own the most patents?)
q2 = """
SELECT 
    name,
    COUNT(DISTINCT patent_id) AS patent_count
FROM companies
WHERE name IS NOT NULL
AND name != ''
GROUP BY company_id
ORDER BY patent_count DESC
LIMIT 20;
"""

df_q2 = pd.read_sql(q2, conn)
print("Q2: TOP COMPANIES")
print("")
print(df_q2.to_string(index=False))

Q2: TOP COMPANIES

                                            name  patent_count
                       SAMSUNG DISPLAY CO., LTD.          2064
     International Business Machines Corporation          1788
                          CANON KABUSHIKI KAISHA          1048
                          SONY GROUP CORPORATION           765
                                 Fujitsu Limited           651
                        Kabushiki Kaisha Toshiba           602
                                   HITACHI, LTD.           583
                               Intel Corporation           581
                 MITSUBISHI ELECTRIC CORPORATION           561
                        General Electric Company           553
                             LG ELECTRONICS INC.           527
                           QUALCOMM Incorporated           502
                 TOYOTA JIDOSHA KABUSHIKI KAISHA           496
              SUMITOMO ELECTRIC INDUSTRIES, LTD.           490
                                 NEC

In [4]:
# Q3 Top Countries (Which countries produce the most patents?)
q3 = """
SELECT 
    country,
    COUNT(DISTINCT patent_id) AS patent_count
FROM inventors
WHERE country IS NOT NULL
AND country != ''
GROUP BY country
ORDER BY patent_count DESC
LIMIT 20;
"""

df_q3 = pd.read_sql(q3, conn)
print("Q3: TOP COUNTRIES")
print("")
print(df_q3.to_string(index=False))

Q3: TOP COUNTRIES

country  patent_count
     US         48234
     JP         16649
     DE          5790
     KR          5275
     CN          4345
     TW          2681
     FR          2038
     GB          2031
     CA          1919
     IN          1106
     IL           971
     CH           878
     IT           831
     NL           783
     SE           735
     AU           525
     BE           380
     FI           371
     AT           357
     DK           321


In [5]:
# Q4 Trends Over Time (How many patents per year?)
q4 = """
SELECT 
    year,
    COUNT(DISTINCT patent_id) AS patent_count
FROM patents
WHERE year IS NOT NULL
GROUP BY year
ORDER BY year ASC;
"""

df_q4 = pd.read_sql(q4, conn)
print("Q4: PATENTS PER YEAR")
print("")
print(df_q4.to_string(index=False))

Q4: PATENTS PER YEAR

 year  patent_count
 2018        100000


In [6]:
# Q5 JOIN Query (Combine patents with inventors and companies)
q5 = """
SELECT 
    p.patent_id,
    p.title,
    p.year,
    i.full_name AS inventor_name,
    i.country AS inventor_country,
    c.name AS company_name
FROM relationships r
JOIN patents p ON r.patent_id = p.patent_id
JOIN inventors i ON r.inventor_id = i.inventor_id
JOIN companies c ON r.company_id = c.company_id
WHERE p.title IS NOT NULL
LIMIT 20;
"""

df_q5 = pd.read_sql(q5, conn)
print("Q5: JOIN QUERY - Patents with Inventors and Companies")
print("")
print(df_q5.to_string(index=False))

Q5: JOIN QUERY - Patents with Inventors and Companies

patent_id                                                 title  year     inventor_name inventor_country                    company_name
 10099298                    Cutting insert and indexable drill  2018      Hyo Joon Lim               KR                     KORLOY INC.
 10099298                    Cutting insert and indexable drill  2018      Hyo Joon Lim               KR                     KORLOY INC.
 10099298                    Cutting insert and indexable drill  2018      Hyo Joon Lim               KR                     KORLOY INC.
 10038114 Semiconductor device and manufacturing method thereof  2018 Shinichi Watanuki               JP Renesas Electronics Corporation
 10038114 Semiconductor device and manufacturing method thereof  2018 Shinichi Watanuki               JP Renesas Electronics Corporation
 10038114 Semiconductor device and manufacturing method thereof  2018 Shinichi Watanuki               JP Renesas Electronic

In [7]:
# Q6 CTE Query (Break complex query into steps)
q6 = """
WITH inventor_counts AS (
    SELECT 
        inventor_id,
        full_name,
        country,
        COUNT(DISTINCT patent_id) AS total_patents
    FROM inventors
    WHERE full_name IS NOT NULL
    GROUP BY inventor_id
),
top_inventors AS (
    SELECT *
    FROM inventor_counts
    WHERE total_patents >= 3
)
SELECT 
    full_name,
    country,
    total_patents
FROM top_inventors
ORDER BY total_patents DESC
LIMIT 20;
"""

df_q6 = pd.read_sql(q6, conn)
print("Q6: CTE QUERY - Inventors with 3 or more patents")
print("")
print(df_q6.to_string(index=False))

Q6: CTE QUERY - Inventors with 3 or more patents

              full_name country  total_patents
       Shunpei Yamazaki      JP             36
        Kia Silverbrook      AU             23
                Tao Luo      US             16
               Junyi Li      US             13
       Bartley K. Andre      US             12
     Duncan Robert Kerr      US             11
       Gurtej S. Sandhu      US             11
               Jing Sun      US             11
  Matthew Dean Rohrbach      US             11
Christopher J. Stringer      US             10
          David R. Hall      US             10
          Kangguo Cheng      US             10
   Peter Russell-Clarke      US             10
      Rico Zörkendörfer      US             10
       Roderick A. Hyde      US             10
             Peter Gaal      US              9
               Shan Liu      US              9
         Takahiro NISHI      JP              9
         Takashi Suzuki      JP              9
          

In [8]:
# Q7 Ranking Query (Rank inventors using window functions)
q7 = """
WITH inventor_counts AS (
    SELECT
        inventor_id,
        full_name,
        country,
        COUNT(DISTINCT patent_id) AS total_patents
    FROM inventors
    WHERE full_name IS NOT NULL
    AND country IS NOT NULL
    GROUP BY inventor_id
),
ranked AS (
    SELECT
        full_name,
        country,
        total_patents,
        RANK() OVER (ORDER BY total_patents DESC) AS overall_rank,
        RANK() OVER (PARTITION BY country ORDER BY total_patents DESC) AS country_rank
    FROM inventor_counts
)
SELECT *
FROM ranked
WHERE overall_rank <= 20
ORDER BY overall_rank;
"""

df_q7 = pd.read_sql(q7, conn)
print("Q7: RANKING QUERY - Inventors ranked overall and by country")
print("")
print(df_q7.to_string(index=False))

Q7: RANKING QUERY - Inventors ranked overall and by country

              full_name country  total_patents  overall_rank  country_rank
       Shunpei Yamazaki      JP             36             1             1
        Kia Silverbrook      AU             23             2             1
                Tao Luo      US             16             3             1
               Junyi Li      US             13             4             2
       Bartley K. Andre      US             12             5             3
     Duncan Robert Kerr      US             11             6             4
       Gurtej S. Sandhu      US             11             6             4
               Jing Sun      US             11             6             4
  Matthew Dean Rohrbach      US             11             6             4
Christopher J. Stringer      US             10            10             8
          David R. Hall      US             10            10             8
          Kangguo Cheng      US        

In [10]:
# Saving all query results
REPORTS_DIR = "../reports/"
import os
os.makedirs(REPORTS_DIR, exist_ok=True)

df_q1.to_csv(REPORTS_DIR + "top_inventors.csv", index=False)
df_q2.to_csv(REPORTS_DIR + "top_companies.csv", index=False)
df_q3.to_csv(REPORTS_DIR + "country_trends.csv", index=False)
df_q4.to_csv(REPORTS_DIR + "patents_per_year.csv", index=False)

print("Query results saved")
print("top_inventors.csv")
print("top_companies.csv")
print("country_trends.csv")
print("patents_per_year.csv")

Query results saved
top_inventors.csv
top_companies.csv
country_trends.csv
patents_per_year.csv


In [11]:
#  Closing the  Connection
conn.close()
print("Database connection closed")


Database connection closed
